# Baseline Evaluation — intent-classifier

This notebook:
1. Clones the `intent-classifier` repo from GitHub
2. Installs required Python packages
3. Authenticates with Hugging Face (needed for gated models like Gemma, Llama)
4. Runs **zero-shot** baseline evaluation across selected models
5. Runs **few-shot** baseline evaluation across selected models
6. Generates analysis plots into `evaluation_baseline/analysis/`
7. Displays the resulting figures inline

> **Runtime:** Set runtime type to **GPU → T4** (or better) for reasonable speed.
> Smaller models (< 0.6 B) run fine on CPU too.

## 1. Clone repository

In [ ]:
import os

REPO_URL = "https://github.com/kon172verma/intent-classifier.git"
REPO_DIR = "/content/intent-classifier"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo already cloned — pulling latest changes.")
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## 2. Install dependencies

In [ ]:
!pip install -q \
    torch \
    transformers \
    accelerate \
    bitsandbytes \
    python-dotenv \
    sentencepiece \
    protobuf \
    matplotlib \
    numpy

print("Dependencies installed.")

## 3. Hugging Face authentication

Required for **gated models** (Gemma-3, Llama-3.2).  
Open-weight models (Qwen, SmolLM, TinyLlama) do **not** need a token.

Store your token in Colab Secrets as **`HF_TOKEN`** (left panel → 🔑 icon)  
or paste it in the cell below.

In [ ]:
import os

# ── Try Colab Secrets first (recommended) ────────────────────────────────────
try:
    from google.colab import userdata  # type: ignore
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        print("HF_TOKEN loaded from Colab Secrets.")
    else:
        print("HF_TOKEN secret not set — gated models will be skipped.")
except Exception:
    if not os.environ.get("HF_TOKEN"):
        print("HF_TOKEN not found — gated models will be skipped.")
    else:
        print("HF_TOKEN already set in environment.")

## 4. Configuration

Edit the lists and settings below to control which models run and how many examples to evaluate.

In [ ]:
import sys

# evaluation_lib lives at repo root — must be on sys.path
sys.path.insert(0, "/content/intent-classifier")
# src scripts are imported directly by the subprocess runner
sys.path.insert(0, "/content/intent-classifier/evaluation_baseline/src")

REPO_DIR   = "/content/intent-classifier"
SRC_DIR    = f"{REPO_DIR}/evaluation_baseline/src"
DATA_FILE  = f"{REPO_DIR}/dataset_sample/sample.json"
ZS_OUT_DIR = f"{REPO_DIR}/evaluation_baseline/reports_zero_shot"
FS_OUT_DIR = f"{REPO_DIR}/evaluation_baseline/reports_few_shot"
PLOT_DIR   = f"{REPO_DIR}/evaluation_baseline/analysis"

# ── Models to evaluate ───────────────────────────────────────────────────────
# Remove any model from the list you want to skip.
# Gated models (gemma3-*, llama3.2-3b) require HF_TOKEN.
MODELS_TO_RUN = [
    "smollm2-135m",
    "smollm2-360m",
    "qwen2.5-0.5b",
    "qwen3-0.6b",
    "tinyllama",
    "gemma3-270m",   # gated — needs HF_TOKEN
    "gemma3-1b",     # gated — needs HF_TOKEN
    "qwen2.5-1.5b",
    "qwen3-1.7b",
    "smollm3",
    "qwen3-4b",
    # "llama3.2-3b",  # gated — uncomment if you have the Meta licence
]

# Set to a small number (e.g. 5) for a quick smoke-test; None = all examples
LIMIT = None

DEVICE = "auto"   # auto | cuda | cpu | mps

print(f"Models to run : {len(MODELS_TO_RUN)}")
print(f"Data file     : {DATA_FILE}")
print(f"Device        : {DEVICE}")
print(f"Limit         : {LIMIT or 'all examples'}")

## 5. Zero-shot evaluation

In [ ]:
import subprocess, sys, os

os.makedirs(ZS_OUT_DIR, exist_ok=True)

run_script = f"{SRC_DIR}/run_all_baselines.py"

cmd = [
    sys.executable, run_script,
    "--models", *MODELS_TO_RUN,
    "--data",    DATA_FILE,
    "--device",  DEVICE,
    "--mode",    "zero_shot",
    "--out-dir", ZS_OUT_DIR,
]
if LIMIT:
    cmd += ["--limit", str(LIMIT)]

print("Running zero-shot evaluation ...\n")
result = subprocess.run(cmd, text=True)
if result.returncode != 0:
    print("\n[ERROR] Zero-shot evaluation failed.")
else:
    print("\nZero-shot evaluation complete.")

## 6. Few-shot evaluation

In [ ]:
os.makedirs(FS_OUT_DIR, exist_ok=True)

cmd = [
    sys.executable, run_script,
    "--models", *MODELS_TO_RUN,
    "--data",    DATA_FILE,
    "--device",  DEVICE,
    "--mode",    "few_shot",
    "--out-dir", FS_OUT_DIR,
]
if LIMIT:
    cmd += ["--limit", str(LIMIT)]

print("Running few-shot evaluation ...\n")
result = subprocess.run(cmd, text=True)
if result.returncode != 0:
    print("\n[ERROR] Few-shot evaluation failed.")
else:
    print("\nFew-shot evaluation complete.")

## 7. Generate analysis plots

In [ ]:
os.makedirs(PLOT_DIR, exist_ok=True)

plot_script = f"{SRC_DIR}/plot_baselines.py"

cmd = [
    sys.executable, plot_script,
    "--reports-dir",          ZS_OUT_DIR,
    "--few-shot-reports-dir", FS_OUT_DIR,
    "--out-dir",              PLOT_DIR,
]

print("Generating plots ...\n")
result = subprocess.run(cmd, text=True)
if result.returncode != 0:
    print("\n[ERROR] Plot generation failed.")
else:
    print("\nPlots saved to:", PLOT_DIR)

## 8. Display results

In [ ]:
import glob
from IPython.display import display, Image
from pathlib import Path

plot_files = sorted(Path(PLOT_DIR).glob("*.png"))

if not plot_files:
    print("No PNG files found in", PLOT_DIR)
else:
    for p in plot_files:
        print(f"\n── {p.name} ──")
        display(Image(filename=str(p)))

## 9. Download outputs (optional)

Zip and download all reports and plots to your local machine.

In [ ]:
import shutil
from google.colab import files  # type: ignore

zip_path = "/content/baseline_results.zip"
shutil.make_archive(
    "/content/baseline_results",
    "zip",
    root_dir=f"{REPO_DIR}/evaluation_baseline",
    base_dir=".",
)
print(f"Archive created: {zip_path}")
files.download(zip_path)